## Setup

In [17]:
# import necessary libraries
import time
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
# load paths
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation import calculate_macro_f1
from src.submission import create_submission

DATA_DIR = PROJECT_ROOT / "data"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_features.csv"
TEST_PATH = DATA_DIR / "test_features.csv"
SPLIT_PATH = DATA_DIR / "splits" / "shared_validation_split.csv"

print(TRAIN_PATH.exists(), TEST_PATH.exists(), SPLIT_PATH.exists())

True True True


In [19]:
# load raw data + shared split
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

ID_COLUMN = "id"
LABEL_COLUMN = "label"

FEATURES = [c for c in train_df.columns if c not in (ID_COLUMN, LABEL_COLUMN)]
assert len(FEATURES) == 5000, f"expected 5000 features, got {len(FEATURES)}"

split = pd.read_csv(SPLIT_PATH)
assert len(split) == len(train_df)

split = split.set_index("row_index")
is_train = (split.loc[train_df.index, "split"] == "train").to_numpy()

X_train = train_df.loc[is_train, FEATURES].to_numpy(dtype=np.float32)
X_val = train_df.loc[~is_train, FEATURES].to_numpy(dtype=np.float32)
y_train = train_df.loc[is_train, LABEL_COLUMN].to_numpy()
y_val = train_df.loc[~is_train, LABEL_COLUMN].to_numpy()

X_train = np.ascontiguousarray(X_train, dtype=np.float32)
X_val = np.ascontiguousarray(X_val, dtype=np.float32)
print(X_train.flags['C_CONTIGUOUS'], X_val.flags['C_CONTIGUOUS'])

print(X_train.shape, X_val.shape)
print(round(y_train.mean(), 4), round(y_val.mean(), 4))

True True
(16000, 5000) (4000, 5000)
0.6252 0.6252


# Complement Naive Bayes (CNB) Model

In [21]:
def cnb_fit(X, y, alpha=1.0, normalize=True):
    classes = np.unique(y)
    n_classes = len(classes)
    n_features = X.shape[1]

    complement_weights = np.zeros((n_classes, n_features))
    total_feature_mass = np.asarray(X.sum(axis=0)).ravel()

    for idx, c in enumerate(classes):
        X_c = X[y == c]
        class_feature_mass = np.asarray(X_c.sum(axis=0)).ravel()
        complement_mass = total_feature_mass - class_feature_mass + alpha
        complement_total = complement_mass.sum()

        weights = np.log(complement_mass / complement_total)
        if normalize:
            weights = weights / np.sum(np.abs(weights))

        complement_weights[idx] = weights

    return classes, complement_weights

In [22]:
def cnb_predict_scores(X, complement_weights):
    return X @ complement_weights.T


def cnb_predict(X, classes, complement_weights):
    scores = cnb_predict_scores(X, complement_weights)
    return classes[np.argmin(scores, axis=1)]


def cnb_predict_proba(X, complement_weights):
    scores = cnb_predict_scores(X, complement_weights)
    neg_scores = -scores
    neg_scores -= neg_scores.max(axis=1, keepdims=True)
    proba = np.exp(neg_scores)
    return proba / proba.sum(axis=1, keepdims=True)

In [23]:
# find best 'alpha' value
cnb_results = []

for alpha in [0.0001, 0.001, 0.01, 0.1, 0.5, 1, 2]:
    classes, complement_weights = cnb_fit(X_train, y_train, alpha=alpha)
    preds = cnb_predict(X_val, classes, complement_weights)
    score = calculate_macro_f1(y_val, preds)
    cnb_results.append({"alpha": alpha, "val_macro_f1": score})
    print(f"alpha={alpha:<6} CNB F1={score:.4f}")

cnb_results_df = pd.DataFrame(cnb_results).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
cnb_results_df

alpha=0.0001 CNB F1=0.6416
alpha=0.001  CNB F1=0.6382
alpha=0.01   CNB F1=0.6324
alpha=0.1    CNB F1=0.6350
alpha=0.5    CNB F1=0.6502
alpha=1      CNB F1=0.6603
alpha=2      CNB F1=0.6669


,alpha,val_macro_f1
0,2.0000,0.666886
1,1.0000,0.660279
2,0.5000,0.650241
3,0.0001,0.641556
4,0.0010,0.638243
5,0.1000,0.634969
6,0.0100,0.632382


In [24]:
best_alpha = float(cnb_results_df.iloc[0].alpha)
print("best alpha:", best_alpha, "val Macro F1:", round(float(cnb_results_df.iloc[0].val_macro_f1), 6))

best alpha: 2.0 val Macro F1: 0.666886


In [26]:
# refit model on all labelled data 
X_full = np.concatenate([X_train, X_val], axis=0)
y_full = np.concatenate([y_train, y_val], axis=0)

final_classes, final_feature_log_prob = cnb_fit(X_full, y_full, alpha=best_alpha)

In [27]:
X_test = test_df[FEATURES].to_numpy(dtype=np.float32)
test_preds = cnb_predict(X_test, final_classes, final_feature_log_prob)

print(pd.Series(test_preds).value_counts().sort_index())

X_full = np.ascontiguousarray(X_full, dtype=np.float32)
X_test = np.ascontiguousarray(X_test, dtype=np.float32)
print(X_full.shape, X_test.shape)

0    1518
1    5481
dtype: int64
(20000, 5000) (6999, 5000)


In [28]:
# save data 
cnb_submission_path = SUBMISSIONS_DIR / "CNB_updated_Prediction.csv"
cnb_submission_df = create_submission(
    test_ids=test_df[ID_COLUMN],
    predictions=test_preds,
    output_path=cnb_submission_path,
    id_column=ID_COLUMN,
    label_column=LABEL_COLUMN,
)
display(cnb_submission_df.head())

,id,label
0,59218,1
1,37110,1
2,23200,1
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,1


In [31]:
saved = pd.read_csv(cnb_submission_path, dtype={ID_COLUMN: "string"})
expected_ids = test_df[ID_COLUMN].astype("string").reset_index(drop=True)

assert saved.columns.tolist() == [ID_COLUMN, LABEL_COLUMN]
assert len(saved) == len(test_df)
assert saved[ID_COLUMN].tolist() == expected_ids.tolist()
assert saved[LABEL_COLUMN].isnull().sum() == 0
assert set(saved[LABEL_COLUMN].unique()).issubset(set(train_df[LABEL_COLUMN].unique()))

print("verified - ready to upload")

verified - ready to upload
